# Wind Forecast — Benchmark v2

Chạy 4 kiến trúc (RF, LSTM, Transformer, Mamba-S6) trên bộ dữ liệu NREL SEA WTK v3.

**Chuẩn bị trên Google Drive trước khi chạy** — tạo thư mục `MyDrive/WIND_FORECAST_DATA/` gồm:

```
WIND_FORECAST_DATA/
    data_v2/     <- 4 file .npz (chạy scripts/05_pack_for_colab.py ở máy để tạo)
    scripts/     <- toàn bộ file .py trong thư mục scripts/
```

Kết quả sẽ được ghi ngược về `WIND_FORECAST_DATA/results/`.

**Nhớ bật GPU:** Runtime → Change runtime type → T4 GPU.

## 1. Kết nối Drive và kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print('CANH BAO: khong co GPU. Runtime -> Change runtime type -> T4 GPU')

## 2. Sao chép code và dữ liệu về đĩa cục bộ

Đọc trực tiếp từ Drive qua FUSE rất chậm. Copy về `/content` rồi mới chạy.

In [ ]:
import os, shutil, glob

DRIVE = '/content/drive/MyDrive/WIND_FORECAST_DATA'
WORK  = '/content/wind_forecast'

os.makedirs(WORK, exist_ok=True)
for sub in ('scripts', 'data_v2'):
    src, dst = f'{DRIVE}/{sub}', f'{WORK}/{sub}'
    if not os.path.isdir(src):
        raise SystemExit(f'Thiếu {src} trên Drive')
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
    n = len(os.listdir(dst))
    mb = sum(os.path.getsize(p) for p in glob.glob(f'{dst}/*'))/1e6
    print(f'{sub:10s} {n} file, {mb:.1f} MB')

os.makedirs(f'{WORK}/results', exist_ok=True)
os.chdir(WORK)
print('\ncwd =', os.getcwd())

## 3. Kiểm định trước khi chạy

Hai bước này bắt lỗi sớm. Nếu bước nào báo lỗi thì **dừng lại**, đừng chạy tiếp.

In [ ]:
# Hợp đồng dữ liệu + baseline persistence (mức sàn mọi model phải vượt)
!python scripts/04_data_pipeline.py

In [ ]:
# Self-test Mamba S6: scan song song vs tuần tự, tính nhân quả, A âm
!python scripts/06_models.py

## 4. Chạy thử 1 seed

Đây là bước quan trọng nhất: trả lời câu hỏi **Mamba có còn thắng không** khi dữ
liệu đã trung thực và Transformer đã về đúng kích thước (71K thay vì 566K tham số).

Mất khoảng 2–3 giờ. Chạy trước khi tốn thời gian cho 5 seed, vì mọi bước sau
(LOSO, SHAP, Table 8) đều là hệ quả của câu trả lời này.

In [ ]:
!python scripts/07_run_benchmark.py --models LSTM,Transformer,Mamba,PatchTST,DLinear --seeds 1 --epochs 50 --patience 10 --out results/pilot.csv

In [ ]:
import pandas as pd
p = pd.read_csv('results/pilot.csv')
print(p.pivot_table(index=['site','horizon'], columns='model', values='RMSE')
       .round(4).to_string())
print()
print('RMSE trung bình toàn bộ:')
print(p.groupby('model').RMSE.mean().sort_values().round(4).to_string())

## 5. Chạy đầy đủ 5 seed

Chỉ chạy sau khi bước 4 cho kết quả hợp lý.

**Thời gian dự kiến trên T4.** Mamba nặng hơn LSTM/Transformer khoảng 7× vì
selective scan phải giữ tensor `(batch, 96, 128, 16)` qua 7 bước quét. Ước tính
20–40 phút mỗi lượt Mamba, 5–10 phút mỗi lượt LSTM/Transformer, ~1,5 phút mỗi
lượt RF. Tổng 80 lượt (4 model × 4 trạm × 5 seed) vào khoảng **8–14 giờ**.

Kết quả được ghi **nối tiếp** vào `results/runs.csv` sau mỗi lượt, và lần chạy
sau sẽ **tự bỏ qua** những lượt đã xong. Nên cứ chạy ô này nhiều phiên — Colab
ngắt giữa chừng cũng không mất gì. Nhớ chạy ô mục 6 để lưu về Drive trước khi
runtime kết thúc.

Nếu hết VRAM: giảm `--batch-size` xuống 64, hoặc thêm `--models` để chạy từng
model một.

In [ ]:
!python scripts/07_run_benchmark.py --models LSTM,Transformer,Mamba,PatchTST,DLinear --seeds 5 --epochs 50 --patience 10 --out results/runs.csv

## 6. Ghi kết quả về Drive

Chạy ô này **thường xuyên** — `/content` sẽ bị xoá khi runtime kết thúc.

In [ ]:
os.makedirs(f'{DRIVE}/results', exist_ok=True)
for f in glob.glob(f'{WORK}/results/*.csv'):
    shutil.copy(f, f'{DRIVE}/results/')
    print('->', os.path.basename(f))
print('\nĐã lưu về Drive.')

## 7. Tổng hợp

In [ ]:
import pandas as pd, numpy as np
r = pd.read_csv('results/runs.csv')
b = pd.read_csv('results/baselines.csv')
b = b[b.model=='persistence']

agg = (r.groupby(['model','horizon']).RMSE
        .agg(['mean','std']).round(4))
print('RMSE trung bình toàn bộ trạm (5 seed):')
print(agg.to_string())
print()
print('So với persistence, theo trạm x horizon:')
piv = r.pivot_table(index=['site','horizon'], columns='model', values='RMSE')
piv['persistence'] = b.set_index(['site','horizon']).RMSE
print(piv.round(4).to_string())